# Los tres grafos, paso a paso: HippoRAG 2, CatRAG y CatRAG-Reif

Este cuaderno compara **sobre artefactos reales** la representación y el proceso de
consulta de los tres sistemas, usando una misma pregunta conductora del benchmark
2WikiMultihopQA. Estructura:

1. **El grafo de tripletes de HippoRAG 2** (cargado de su `graph.pickle` oficial) y su
   paseo, **re-ejecutado en vivo** con su propia mecánica (PPR estático de igraph).
2. **CatRAG**: mismo grafo; qué re-pondera y por qué no basta (código no liberado —
   mecanismo según su artículo, cifras publicadas con nuestro mismo backbone).
3. **Nuestro grafo canónico de aserciones** y la consulta completa paso a paso
   (afinidad → enlazador → analista → siembra → PPR modulado → salto → conjunto).

Requisitos: ejecutar con el *kernel del venv* del proyecto (tiene `igraph`, `pandas`,
`networkx`). Coste de API ≈ 0: la llamada del analista está cacheada y solo se paga
el embedding de la pregunta (~0,0001 \$).

In [1]:
import json, pickle, re, sys
import numpy as np
sys.path.insert(0, '../src')
RAIZ = '../artifacts/wiki2'
HIPPO = f'{RAIZ}/hipporag_benchmark/gpt-4o-mini_text-embedding-3-small'

corpus = {c['title']: c['text'] for c in json.load(open('../HippoRAG/reproduce/dataset/2wikimultihopqa_corpus.json'))}
preguntas = {q['_id']: q for q in json.load(open('../HippoRAG/reproduce/dataset/2wikimultihopqa.json'))}
res_hippo = {json.loads(l)['_id']: json.loads(l) for l in open(f'{RAIZ}/resultados_hipporag2_benchmark.jsonl')}
res_reif  = {json.loads(l)['_id']: json.loads(l) for l in open(f'{RAIZ}/resultados_plan_v4_deepseek_benchmark.jsonl')} \
            if __import__('os').path.exists(f'{RAIZ}/resultados_plan_v4_deepseek_benchmark.jsonl') else \
            {json.loads(l)['_id']: json.loads(l) for l in open(f'{RAIZ}/resultados_plan_deepseek_benchmark.jsonl')}

QID = next(i for i,q in preguntas.items() if q['question'].startswith('Which film has the director who is older, God'))
q = preguntas[QID]
print('PREGUNTA :', q['question'])
print('ORO      :', [t for t,_ in q['supporting_facts']])
print('CADENA   :', q['evidences'])

PREGUNTA : Which film has the director who is older, God'S Gift To Women or Aldri Annet Enn Bråk?
ORO      : ["God's Gift to Women", 'Aldri annet enn bråk', 'Michael Curtiz', 'Edith Carlmar']
CADENA   : [["God's Gift to Women", 'director', 'Michael Curtiz'], ['Aldri annet enn bråk', 'director', 'Edith Carlmar'], ['Michael Curtiz', 'date of birth', 'December 24, 1886'], ['Edith Carlmar', 'date of birth', '15 November 1911']]


## 1 · El grafo de HippoRAG 2: hechos como aristas, menciones sin resolver

HippoRAG 2 extrae de cada pasaje **tripletes** `(sujeto, relación, objeto)` y los
materializa como **aristas** entre *nodos de frase* (las menciones, en minúsculas),
más aristas de sinonimia por umbral de coseno y aristas de contexto frase→pasaje.
Cargamos su grafo oficial tal como lo construyó su código en nuestra reproducción.

In [2]:
import igraph, pandas as pd, glob
G_h = pickle.load(open(f'{HIPPO}/graph.pickle','rb'))
print(type(G_h).__name__, '| nodos:', G_h.vcount(), '| aristas:', G_h.ecount())
# nombres = hashes; el texto vive en los parquet (hash_id -> content)
ent = pd.concat([pd.read_parquet(p) for p in glob.glob(f'{HIPPO}/entity_embeddings/*.parquet')])
chk = pd.concat([pd.read_parquet(p) for p in glob.glob(f'{HIPPO}/chunk_embeddings/*.parquet')])
h2frase = dict(zip(ent['hash_id'], ent['content']))
h2chunk = dict(zip(chk['hash_id'], chk['content']))
titulo = lambda h: h2chunk.get(h, '').split(chr(10),1)[0][:60]   # 1a linea del pasaje = titulo
print('frases-entidad:', len(h2frase), '| pasajes:', len(h2chunk))
print('ejemplo de nodo:', G_h.vs['name'][0], '->', repr(h2frase.get(G_h.vs['name'][0], '(pasaje)')))

Graph | nodos: 56127 | aristas: 305826


frases-entidad: 50008 | pasajes: 6119
ejemplo de nodo: entity-d41d8cd98f00b204e9800998ecf8427e -> ''


In [3]:
# Fragmentacion de identidad: variantes de una misma persona como nodos DISTINTOS
var = sorted({f for f in h2frase.values() if 'carlmar' in f.lower() or 'curtiz' in f.lower()})
print('frases-entidad que contienen curtiz/carlmar (cada una es un nodo aparte):')
for v in var: print('  -', repr(v))

frases-entidad que contienen curtiz/carlmar (cada una es un nodo aparte):
  - 'carlmar film a s'
  - 'edith carlmar'
  - 'edith carlmar to cinema'
  - 'michael curtiz'
  - 'otto carlmar'


In [4]:
# Los hechos-arista: relacion como atributo textual de la arista (no un nodo)
er = [a for a in G_h.es.attributes() if a != 'weight']
print('atributos de arista:', G_h.es.attributes())
def aristas_con(pat):
    out=[]
    for e in G_h.es:
        s, t = h2frase.get(G_h.vs[e.source]['name'],''), h2frase.get(G_h.vs[e.target]['name'],'')
        rel = e[er[0]] if er else ''
        if pat in (s+' '+str(rel)+' '+t).lower(): out.append(f'{s} --[{rel}]--> {t}')
    return out
print("aristas con \"god's gift\":")
for a in aristas_con("god's gift")[:6]: print('  ', a)

atributos de arista: ['weight']
aristas con "god's gift":


**Lectura.** El hecho puente existe (*god's gift to women — directed by — michael
curtiz*), pero es una **arista anónima**: sin frase propia, sin procedencia
direccionable, y con las menciones tal cual salieron del NER. Su proceso de consulta:
(1) coseno pregunta→tripletes; (2) filtro LLM (*recognition memory*, ≤4 tripletes);
(3) las frases de los tripletes aprobados son las **semillas**; (4) todos los pasajes
reciben un prior denso ×0,05; (5) **PPR con matriz estática** (λ=0,5). Vamos a
re-ejecutar los pasos 3–5 con su propio grafo para ver la deriva.

In [5]:
# Re-ejecucion de la mecanica de HippoRAG: sembrar SOLO la rama que su filtro cubrio
# (Carlmar + los dos films) y correr su PPR estatico sobre SU grafo.
idx = {n:i for i,n in enumerate(G_h.vs['name'])}
# hash de las frases-semilla
frase2hash = {v:k for k,v in h2frase.items()}
sem_frases = [f for f in h2frase.values() if f.lower() in
              ("god's gift to women","aldri annet enn braak","aldri annet enn bråk","edith carlmar")]
sem = [idx[frase2hash[f]] for f in sem_frases if frase2hash.get(f) in idx]
print('semillas (rama de Carlmar):', sem_frases)
reset=[0.0]*G_h.vcount()
for i in sem: reset[i]=1.0
w = G_h.es['weight'] if 'weight' in G_h.es.attributes() else None
pr = G_h.personalized_pagerank(damping=0.5, reset=reset, weights=w, implementation='prpack')
oro = {t for t,_ in q['supporting_facts']}
pas = sorted(((titulo(G_h.vs[i]['name']), pr[i]) for i in range(G_h.vcount())
              if G_h.vs[i]['name'].startswith('chunk')), key=lambda x:-x[1])
print('\ntop-8 pasajes del paseo estatico con esa siembra:')
for t,p in pas[:8]:
    print(f'  {p:.5f}  {t}' + ('  <== ORO' if any(o.lower() in t.lower() for o in oro) else ''))

semillas (rama de Carlmar): ['edith carlmar']



top-8 pasajes del paseo estatico con esa siembra:
  0.02918  Edith Carlmar  <== ORO
  0.00706  Aldri annet enn bråk  <== ORO
  0.00662  Altid ballade
  0.00619  Bedre enn sitt rykte
  0.00055  Karl-Ludvig Bugge
  0.00054  Iván Noel
  0.00038  Eva Seeberg
  0.00035  The Parson and the Outlaw


In [6]:
# Y lo que su sistema COMPLETO devolvio en la medicion oficial (filtro incluido):
r = res_hippo[QID]
print('top-5 medido de HippoRAG 2:')
for k, t in enumerate(r['top'][:5]):
    print(f"  {k+1}. {t}" + ('  <== ORO' if t in oro else '  <== deriva (otra pelicula de Carlmar)'))
print(f"R@5 = {r['recall@5']:.2f}   FC@5 = {int(r['full_chain@5'])}")

top-5 medido de HippoRAG 2:
  1. Aldri annet enn bråk  <== ORO
  2. God's Gift to Women  <== ORO
  3. Edith Carlmar  <== ORO
  4. Altid ballade  <== deriva (otra pelicula de Carlmar)
  5. Bedre enn sitt rykte  <== deriva (otra pelicula de Carlmar)
R@5 = 0.75   FC@5 = 0


**Diagnóstico.** La masa difunde por el vecindario más denso de las semillas — las
*otras películas* de Edith Carlmar — y **Michael Curtiz jamás se siembra**: su filtro
(≤4 tripletes) cubrió una sola rama, y con matriz estática ninguna consulta puede
redistribuir lo que la siembra no puso. Firma característica: exhaustividad parcial
alta (3 de 4 oros) con cadena rota (FC=0).

## 2 · CatRAG: re-ponderar el mismo grafo

CatRAG (código no liberado; mecanismo y cifras según su artículo, publicadas con
**nuestro mismo backbone** gpt-4o-mini + te3-small) construye sobre el grafo de
HippoRAG 2 **sin cambiar nada en indexación** y añade tres mecanismos de consulta:
anclas simbólicas débiles (ε), re-ponderación por LLM de las aristas **cercanas a
las semillas**, y refuerzo de pasajes con hechos verificados. Su límite es visible en
este ejemplo: hereda la siembra y la identidad — *no puede crear la semilla ausente*
(Curtiz) ni fusionar variantes de mención; re-ponderar aristas próximas a semillas no
alcanza a un nodo que no está cerca de ninguna. Resultado publicado en 2Wiki:
**R@5 87,0 / FCR 67,6** (+1,1/+1,5 sobre su base — la mejora «modesta» que su propio
resumen admite).

## 3 · Nuestro grafo: aserciones-nodo sobre entidades canónicas

Dos decisiones de índice: (i) cada hecho es un **nodo** con frase autocontenida
`τ_a`, roles n-arios y procedencia; (ii) las menciones se resuelven ANTES de
construir el grafo en una **memoria canónica** (alias, descripción, *pasaje propio*)
— cero aristas de sinonimia. Cargamos el retriever completo (tarda ~1-2 min: lee el
grafo de 84k nodos y los vectores).

In [7]:
from asistente_vih.retrieval.wiki2_plan import Wiki2PlanRAG
r4 = Wiki2PlanRAG(split='benchmark', modelo='deepseek-chat')
from collections import Counter
print('nodos por tipo :', dict(Counter(d.get('tipo') for _, d in r4.g.nodes(data=True))))
print('aristas por tipo:', dict(Counter(d.get('tipo_relacion') for _,_,d in r4.g.edges(data=True))))

nodos por tipo : {'Chunk': 6119, 'Entidad': 26940, 'Asercion': 47999}
aristas por tipo: {'MENCIONADO_EN': 36811, 'EN_CHUNK': 47999, 'HAS_INTERVENTION': 47466, 'HAS_OUTCOME': 32964}


In [8]:
# La asercion puente como NODO de primera clase
aid = next(a for a in r4.aser_ids
           if 'curtiz' in r4.g.nodes[a].get('descripcion','').lower()
           and "god's gift" in r4.g.nodes[a].get('descripcion','').lower())
print('id        :', aid)
print('frase τ_a :', r4.g.nodes[aid]['descripcion'])
print('relacion  :', r4.g.nodes[aid].get('relacion_base'))
print('aristas del nodo-hecho:')
for _, v, d in r4.g.out_edges(aid, data=True):
    print(f"   --{d['tipo_relacion']:>16}--> {v}")

id        : God's Gift to Women::a1
frase τ_a : God's Gift to Women was directed by Michael Curtiz.
relacion  : DIRECTED
aristas del nodo-hecho:
   --        EN_CHUNK--> chunk:God's Gift to Women
   --HAS_INTERVENTION--> can:god's gift to women
   --     HAS_OUTCOME--> can:michael curtiz


In [9]:
# La entrada canonica: identidad resuelta en el indice
for eid in ('can:michael curtiz', 'can:edith carlmar'):
    e = r4.entradas.get(eid)
    print(eid, '->', {k: e[k] for k in ('nombre','alias','pasaje_propio','n_menciones')})

can:michael curtiz -> {'nombre': 'Michael Curtiz', 'alias': ['Michael Curtiz'], 'pasaje_propio': 'Michael Curtiz', 'n_menciones': 9}
can:edith carlmar -> {'nombre': 'Edith Carlmar', 'alias': ['Carlmar Film A/S', 'Edith Carlmar'], 'pasaje_propio': 'Edith Carlmar', 'n_menciones': 5}


Compárese con la sección 1: donde HippoRAG tenía variantes de mención sueltas y una
arista anónima, aquí hay **un** nodo por persona (con sus alias y su *pasaje propio*)
y un nodo-hecho direccionable con su frase. Ahora, la consulta paso a paso.

## 4 · La consulta de CatRAG-Reif, paso a paso

In [10]:
# PASO 1 - afinidad global pregunta-hecho (un producto matricial)
qv = r4._qvecs(q['question'], 'lite')[0]
sims = r4.aser_emb @ qv
print('top-6 hechos por coseno:')
for j in np.argsort(-sims)[:6]:
    print(f"  σ={sims[j]:.3f}  {r4.g.nodes[r4.aser_ids[int(j)]]['descripcion'][:95]}")

top-6 hechos por coseno:
  σ=0.554  God's Gift to Women was directed by Michael Curtiz.
  σ=0.517  God's Gift to Women was originally completed as a musical film.
  σ=0.516  God's Gift to Women was released in 1931.
  σ=0.507  God's Gift to Women stars Joan Blondell.
  σ=0.480  Edith Carlmar is known for the film 'Aldri annet enn bråk'.
  σ=0.477  God's Gift to Women is based on the play The Devil Was Sick.


In [11]:
# PASO 2 - enlazador: menciones de la pregunta -> entradas canonicas
enlaces = r4._enlazar_menciones(q['question'])
print('L(q) =', [(e, r4.entradas[e]['pasaje_propio']) for e in enlaces])

L(q) = [("can:god's gift to women", "God's Gift to Women"), ('can:aldri annet enn bråk', 'Aldri annet enn bråk')]


In [12]:
# PASO 3 - analista (cacheado: coste 0): plan + seleccion + huecos
aprobados, huecos, enlaces2, plan = r4._analista(q['question'], sims, enlaces)
print('PLAN:');  [print('   -', p) for p in plan]
print('SELECCIONADOS:')
for j in aprobados: print('   *', r4.g.nodes[r4.aser_ids[j]]['descripcion'][:95])
print('HUECOS SIN CUBRIR:', huecos)

PLAN:
   - director of God's Gift to Women -> Michael Curtiz
   - date of birth of Michael Curtiz
   - director of Aldri annet enn bråk -> Edith Carlmar
   - date of birth of Edith Carlmar
SELECCIONADOS:
   * God's Gift to Women was directed by Michael Curtiz.
   * Aldri annet enn bråk was edited by Edith Carlmar.
HUECOS SIN CUBRIR: [{'query': 'date of birth of Michael Curtiz', 'anchor': 'Michael Curtiz'}, {'query': 'date of birth of Edith Carlmar', 'anchor': 'Edith Carlmar'}]


In [13]:
# PASO 4 - siembra v (compuerta dura + pasajes propios + suelo denso + anclas)
st = sims.copy(); st[aprobados] = 1.0
r4._enlaces_actuales = enlaces2
v = r4._semillas(q['question'], [qv], st, aprobados)
print('componentes de v (entidades y pasajes con mas masa):')
for n, w in sorted(v.items(), key=lambda x: -x[1])[:10]:
    print(f'   {w:6.3f}  {n}')

componentes de v (entidades y pasajes con mas masa):
    1.050  can:god's gift to women
    1.050  can:michael curtiz
    1.000  can:aldri annet enn bråk
    1.000  can:edith carlmar
    0.575  chunk:God's Gift to Women
    0.555  chunk:Michael Curtiz
    0.546  chunk:Edith Carlmar
    0.545  chunk:Aldri annet enn bråk
    0.046  chunk:Winter Light
    0.046  chunk:Through a Glass Darkly (film)


In [14]:
# PASOS 5-6 - PPR modulado (μ=1 en los hechos aprobados) y ranking
p = r4._ppr(v, st)
orden = np.argsort(-p)
top = [r4.nodos[i][6:] for i in orden if r4.nodos[i].startswith('chunk:')][:10]
print('top-10 pasajes tras el paseo:')
for k, t in enumerate(top):
    print(f"  {k+1:2d}. {t}" + ('  <== ORO' if t in oro else ''))

top-10 pasajes tras el paseo:
   1. God's Gift to Women  <== ORO
   2. Edith Carlmar  <== ORO
   3. Aldri annet enn bråk  <== ORO
   4. Michael Curtiz  <== ORO
   5. Altid ballade
   6. Bedre enn sitt rykte
   7. Mrs. Dane's Confession
   8. Prisoner of the Night (film)
   9. The Lady Takes a Sailor
  10. The Vagabond King (1956 film)


In [15]:
# PASOS 7-8 - salto dirigido (si hubo huecos) y conjunto final; metrica
final = r4.buscar(q['question'], 5)
print('CONJUNTO FINAL (top-5):')
for k, t in enumerate(final): print(f"  {k+1}. {t}" + ('  <== ORO' if t in oro else ''))
print('FC@5 =', int(set(oro) <= set(final)))

CONJUNTO FINAL (top-5):
  1. God's Gift to Women  <== ORO
  2. Michael Curtiz  <== ORO
  3. Aldri annet enn bråk  <== ORO
  4. Edith Carlmar  <== ORO
  5. Altid ballade
FC@5 = 1


## 5 · Comparativa final

| etapa | HippoRAG 2 | CatRAG | CatRAG-Reif |
|---|---|---|---|
| índice | tripletes-arista + sinonimia; menciones sin resolver | **el mismo** | aserciones-nodo sobre entidades canónicas |
| siembra | frases de ≤4 tripletes filtrados (una rama) | **la misma** + anclas ε | plan → compuerta en *ambas* cadenas + pasajes propios |
| paseo | matriz estática | re-pondera aristas cerca de las semillas | μ por hecho global + salto por huecos + conjunto |
| esta pregunta | FC@5 = 0 (deriva a Carlmar) | — (mecanismo heredado) | FC@5 = 1 (4/4 en top-4) |
| benchmark (FC@5) | 65,8 | 67,6 (publicado) | **97,7** |

**Conclusión didáctica**: CatRAG cambia *una* etapa (el paseo) y hereda las demás;
por eso su mejora es marginal — *re-ponderar no alcanza a una semilla que no existe*.
El salto de calidad viene de cambiar el índice (representación + identidad) y de
asignar los recursos de consulta **por hueco del plan**, no por consulta. Detalle
completo: `artifacts/wiki2/INFORME.md`, `docs/presentacion_sistema.pdf` y
`docs/paper_draft_en.pdf`.